# Ridge & Lasso Regression

**Course**: CMOR 438 / INDE 577 — Data Science & Machine Learning  
**Dataset**: International Football Results (1872–present)  
**Author**: Neriah29  

---

## Motivation — The Problem with Plain Linear Regression

Linear Regression minimizes MSE with no constraints on how large weights can get. Given enough features, it will find weights that fit the training data perfectly — including all the noise. This is overfitting.

The fix: **add a penalty for large weights directly into the loss function.** Force the model to keep weights small, which means simpler, more generalizable solutions.

---

## Ridge Regression (L2)

$$\text{Loss} = \underbrace{\frac{1}{n}\sum(y - \hat{y})^2}_{\text{MSE}} + \underbrace{\alpha \sum w_j^2}_{\text{L2 penalty}}$$

The L2 penalty adds the **sum of squared weights**. This:
- Shrinks all weights toward zero proportionally
- Never drives any weight to exactly zero
- All features stay in the model, just with smaller influence

The gradient gets one extra term: `+ 2α * w` — a force pulling every weight back toward zero at every update.

---

## Lasso Regression (L1)

$$\text{Loss} = \underbrace{\frac{1}{n}\sum(y - \hat{y})^2}_{\text{MSE}} + \underbrace{\alpha \sum |w_j|}_{\text{L1 penalty}}$$

The L1 penalty adds the **sum of absolute weights**. This:
- Can shrink weights all the way to exactly zero
- Performs **automatic feature selection** — irrelevant features are dropped
- Produces sparse models — fewer features, easier to interpret

The subgradient is `sign(w)` — the same absolute force regardless of weight size, so small weights get pushed to zero.

---

## Alpha — Regularization Strength

| α | Effect |
|---|---|
| 0 | Plain Linear Regression — no penalty |
| Small | Mild regularization — close to Linear Regression |
| Large | Strong shrinkage — weights pushed heavily toward zero |
| Very large | Underfitting — weights forced near zero, model too simple |


---
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import sys
sys.path.insert(0, '../../src')
from football_ml.supervised_learning.ridge_lasso import RidgeRegression, LassoRegression
from football_ml.supervised_learning.linear_regression import LinearRegression

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

---
## 1. Regularization Effect — Toy Example

Let's see how L1 and L2 penalties behave on a dataset where only 2 of 10 features truly matter.

In [ ]:
rng = np.random.default_rng(0)
X_toy = rng.normal(size=(300, 10))
# Only features 0 and 1 matter
y_toy = 3 * X_toy[:, 0] + 2 * X_toy[:, 1] + rng.normal(scale=0.2, size=300)

scaler_toy = StandardScaler()
X_toy_sc = scaler_toy.fit_transform(X_toy)

lr   = LinearRegression(learning_rate=0.1, n_epochs=1000).fit(X_toy_sc, y_toy)
ridge = RidgeRegression(alpha=1.0, learning_rate=0.05, n_epochs=1000).fit(X_toy_sc, y_toy)
lasso = LassoRegression(alpha=0.5, learning_rate=0.05, n_epochs=1000).fit(X_toy_sc, y_toy)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(10)
width = 0.25
ax.bar(x - width, lr.weights_,    width, label='Linear Regression', color='#95a5a6')
ax.bar(x,         ridge.weights_, width, label='Ridge (α=1.0)',      color='#4878CF')
ax.bar(x + width, lasso.weights_, width, label='Lasso (α=0.5)',      color='#E87272')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f'w{i}' for i in range(10)])
ax.set_ylabel('Weight value', fontsize=12)
ax.set_title('Weight Comparison — Linear vs Ridge vs Lasso\n(only features 0 and 1 truly matter)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'Lasso zero weights: {lasso.n_zero_weights} of 10')

### What to notice

- **Linear Regression**: all weights are nonzero, including the 8 irrelevant features — it fitted noise
- **Ridge**: all weights shrunk, but none are zero — still 10 features in the model
- **Lasso**: irrelevant weights driven to (near) zero — automatic feature selection, only the truly useful features survive

---
## 2. Load & Engineer Features

In [ ]:
df = pd.read_csv('../../data/results.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Target: goal difference (regression target, same as Linear Regression notebook)
df['goal_diff'] = df['home_score'] - df['away_score']

def compute_team_rolling_stats(df, window=10):
    home_log = df[['date', 'home_team', 'home_score', 'away_score']].copy()
    home_log.columns = ['date', 'team', 'scored', 'conceded']
    away_log = df[['date', 'away_team', 'away_score', 'home_score']].copy()
    away_log.columns = ['date', 'team', 'scored', 'conceded']
    team_log = pd.concat([home_log, away_log]).sort_values('date').reset_index(drop=True)
    team_log['rolling_scored'] = (
        team_log.groupby('team')['scored']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    team_log['rolling_conceded'] = (
        team_log.groupby('team')['conceded']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    return team_log.drop_duplicates(subset=['date', 'team'], keep='last').set_index(['date', 'team'])

team_stats = compute_team_rolling_stats(df)

def get_stat(row, team_col, stat_col):
    try:
        return team_stats.loc[(row['date'], row[team_col]), stat_col]
    except KeyError:
        return np.nan

df['home_goals_rolling']    = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_scored'), axis=1)
df['home_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_conceded'), axis=1)
df['away_goals_rolling']    = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_scored'), axis=1)
df['away_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_conceded'), axis=1)

home_wins = df.groupby('home_team').apply(lambda g: (g['home_score'] > g['away_score']).mean()).rename('home_win_rate')
away_wins = df.groupby('away_team').apply(lambda g: (g['away_score'] > g['home_score']).mean()).rename('away_win_rate')
df = df.join(home_wins, on='home_team').join(away_wins, on='away_team')
df['neutral'] = df['neutral'].astype(int)

feature_cols = [
    'home_goals_rolling', 'away_goals_rolling',
    'home_conceded_rolling', 'away_conceded_rolling',
    'home_win_rate', 'away_win_rate', 'neutral'
]
df_clean = df[feature_cols + ['goal_diff']].dropna()
print(f'Dataset: {df_clean.shape[0]} matches')

---
## 3. Prepare Data

In [ ]:
X = df_clean[feature_cols].values
y = df_clean['goal_diff'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')

---
## 4. Train All Three Models

In [ ]:
lr_model    = LinearRegression(learning_rate=0.05, n_epochs=1000, random_state=SEED).fit(X_train_sc, y_train)
ridge_model = RidgeRegression(alpha=1.0, learning_rate=0.05, n_epochs=1000, random_state=SEED).fit(X_train_sc, y_train)
lasso_model = LassoRegression(alpha=0.1, learning_rate=0.05, n_epochs=1000, random_state=SEED).fit(X_train_sc, y_train)

print(f'{"Model":<22} {"Train R²":>10} {"Test R²":>10} {"Test RMSE":>12}')
print('-' * 56)
for name, m in [('Linear Regression', lr_model), ('Ridge (α=1.0)', ridge_model), ('Lasso (α=0.1)', lasso_model)]:
    print(f'{name:<22} {m.score(X_train_sc, y_train):>10.4f} {m.score(X_test_sc, y_test):>10.4f} {m.rmse(X_test_sc, y_test):>12.4f}')

---
## 5. Weight Comparison on Football Data

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(feature_cols))
width = 0.25

ax.bar(x - width, lr_model.weights_,    width, label='Linear Regression', color='#95a5a6')
ax.bar(x,         ridge_model.weights_, width, label='Ridge (α=1.0)',      color='#4878CF')
ax.bar(x + width, lasso_model.weights_, width, label='Lasso (α=0.1)',      color='#E87272')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(feature_cols, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Weight value', fontsize=12)
ax.set_title('Learned Weights — Linear vs Ridge vs Lasso', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'Lasso zero weights: {lasso_model.n_zero_weights} of {len(feature_cols)}')

---
## 6. Effect of Alpha — Regularization Path

In [ ]:
alphas = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]

ridge_weights = []
lasso_weights = []

for a in alphas:
    r = RidgeRegression(alpha=a, learning_rate=0.05, n_epochs=500, random_state=SEED).fit(X_train_sc, y_train)
    l = LassoRegression(alpha=a, learning_rate=0.05, n_epochs=500, random_state=SEED).fit(X_train_sc, y_train)
    ridge_weights.append(r.weights_.copy())
    lasso_weights.append(l.weights_.copy())

ridge_weights = np.array(ridge_weights)
lasso_weights = np.array(lasso_weights)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = plt.cm.tab10(np.linspace(0, 1, len(feature_cols)))

for i, (feat, color) in enumerate(zip(feature_cols, colors)):
    axes[0].plot(alphas, ridge_weights[:, i], marker='o', color=color, label=feat, linewidth=1.5)
    axes[1].plot(alphas, lasso_weights[:, i], marker='o', color=color, label=feat, linewidth=1.5)

for ax, title in zip(axes, ['Ridge — Regularization Path', 'Lasso — Regularization Path']):
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xscale('log')
    ax.set_xlabel('α (log scale)', fontsize=12)
    ax.set_ylabel('Weight value', fontsize=12)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=7, loc='upper right')

plt.tight_layout()
plt.show()

### Reading the regularization paths

- **Ridge (left)**: as α increases, all weights shrink smoothly toward zero but never quite reach it
- **Lasso (right)**: as α increases, weights hit zero one by one and stay there — this is feature selection happening in real time

The order in which Lasso zeros weights tells you which features it considers least important.

---
## 7. Choosing Alpha — Test Score vs Alpha

In [ ]:
ridge_r2, lasso_r2 = [], []
for a in alphas:
    r = RidgeRegression(alpha=a, learning_rate=0.05, n_epochs=500, random_state=SEED).fit(X_train_sc, y_train)
    l = LassoRegression(alpha=a, learning_rate=0.05, n_epochs=500, random_state=SEED).fit(X_train_sc, y_train)
    ridge_r2.append(r.score(X_test_sc, y_test))
    lasso_r2.append(l.score(X_test_sc, y_test))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(alphas, ridge_r2, 'o-', color='#4878CF', linewidth=2, label='Ridge')
ax.plot(alphas, lasso_r2, 'o-', color='#E87272', linewidth=2, label='Lasso')
ax.set_xscale('log')
ax.set_xlabel('α (log scale)', fontsize=12)
ax.set_ylabel('Test R²', fontsize=12)
ax.set_title('Test R² vs Regularization Strength', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 8. Discussion — Do Ridge & Lasso Suit Football?

### What they do well
- **Ridge**: more robust than plain Linear Regression when features are correlated — which they are in football
- **Lasso**: automatic feature selection — useful when you suspect only a few features truly matter
- Both prevent the weight explosion that can happen with many correlated features
- Interpretable — weights are readable, just constrained

### What they still struggle with
1. **Still linear**: regularization doesn't add non-linearity — same linear decision surface as before
2. **Football is noisy**: regularization helps generalization but can't solve the fundamental unpredictability of football
3. **Alpha needs tuning**: the right alpha depends on the data — requires cross-validation in practice

### When to choose which
- **Ridge** when you believe all features contribute something and you want stable predictions
- **Lasso** when you suspect many features are irrelevant and want automatic selection
- **Neither** when you need non-linear boundaries — use ensembles or neural networks instead

---
## Summary

| | Linear Regression | Ridge | Lasso |
|---|---|---|---|
| **Penalty** | None | L2: α Σw² | L1: α Σ\|w\| |
| **Weight behavior** | Unconstrained | Shrinks toward 0 | Can reach exactly 0 |
| **Feature selection** | No | No | Yes |
| **Best for** | Baseline | Correlated features | Sparse problems |
| **Key concept** | MSE minimization | L2 regularization | L1 regularization + sparsity |
